## Initial Data Inspection

**Main fights table:** 6,012 rows × 144 columns

Key null issues identified:
- 1,427 nulls in rolling average columns (B_avg_KD, B_avg_SIG_STR_pct, etc.) — expected for fighters with no prior history, keep rows
- 406 nulls in R_Reach_cms — ~7% missing, will impute with median by weight class
- 739 nulls in DOB — age features will have gaps, flag missing values rather than drop
- Date column is string dtype — must convert to datetime before temporal sorting

In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/data.csv')
fighter_details = pd.read_csv('../data/raw/raw_fighter_details.csv')
fight_stats = pd.read_csv('../data/raw/raw_total_fight_data.csv')

print("Main fights table:", df.shape)
print("Fighter details:", fighter_details.shape)
print("Fight stats:", fight_stats.shape)

Main fights table: (6012, 144)
Fighter details: (3596, 14)
Fight stats: (6012, 1)


In [2]:
# Check the actual delimiter
with open('../data/raw/raw_total_fight_data.csv', 'r') as f:
    print(f.readline())

R_fighter;B_fighter;R_KD;B_KD;R_SIG_STR.;B_SIG_STR.;R_SIG_STR_pct;B_SIG_STR_pct;R_TOTAL_STR.;B_TOTAL_STR.;R_TD;B_TD;R_TD_pct;B_TD_pct;R_SUB_ATT;B_SUB_ATT;R_REV;B_REV;R_CTRL;B_CTRL;R_HEAD;B_HEAD;R_BODY;B_BODY;R_LEG;B_LEG;R_DISTANCE;B_DISTANCE;R_CLINCH;B_CLINCH;R_GROUND;B_GROUND;win_by;last_round;last_round_time;Format;Referee;date;location;Fight_type;Winner



In [3]:
fight_stats = pd.read_csv('../data/raw/raw_total_fight_data.csv', sep=';')
print("Fight stats:", fight_stats.shape)

Fight stats: (6012, 41)


In [4]:
print("=== MAIN FIGHTS TABLE ===")
print(df.dtypes)
print("\nNull counts:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\n=== FIGHTER DETAILS ===")
print(fighter_details.dtypes)
print("\nNull counts:")
print(fighter_details.isnull().sum()[fighter_details.isnull().sum() > 0])

print("\n=== SAMPLE ROW (main fights) ===")
df.head(2)

=== MAIN FIGHTS TABLE ===
R_fighter           str
B_fighter           str
Referee             str
date                str
location            str
                 ...   
R_Height_cms    float64
R_Reach_cms     float64
R_Weight_lbs    float64
B_age           float64
R_age           float64
Length: 144, dtype: object

Null counts:
Referee                    32
B_avg_KD                 1427
B_avg_opp_KD             1427
B_avg_SIG_STR_pct        1427
B_avg_opp_SIG_STR_pct    1427
                         ... 
R_Height_cms                4
R_Reach_cms               406
R_Weight_lbs                2
B_age                     172
R_age                      63
Length: 109, dtype: int64

=== FIGHTER DETAILS ===
fighter_name        str
Height              str
Weight              str
Reach               str
Stance              str
DOB                 str
SLpM            float64
Str_Acc             str
SApM            float64
Str_Def             str
TD_Avg          float64
TD_Acc              str


,R_fighter,B_fighter,Referee,date,location,Winner,title_bout,weight_class,B_avg_KD,B_avg_opp_KD,...,R_win_by_Decision_Unanimous,R_win_by_KO/TKO,R_win_by_Submission,R_win_by_TKO_Doctor_Stoppage,R_Stance,R_Height_cms,R_Reach_cms,R_Weight_lbs,B_age,R_age
0,Adrian Yanez,Gustavo Lopez,Chris Tognoni,2021-03-20,"Las Vegas, Nevada, USA",Red,False,Bantamweight,0.0,0.0,...,0,1,0,0,Orthodox,170.18,177.80,135.0,31.0,27.0
1,Trevin Giles,Roman Dolidze,Herb Dean,2021-03-20,"Las Vegas, Nevada, USA",Red,False,Middleweight,0.5,0.0,...,0,3,0,0,Orthodox,182.88,187.96,185.0,32.0,28.0


In [5]:
# Check date range and sort order
df['date'] = pd.to_datetime(df['date'])
print(df['date'].min(), df['date'].max())
print("Already sorted?", df['date'].is_monotonic_increasing)

1994-03-11 00:00:00 2021-03-20 00:00:00
Already sorted? False


In [6]:
df = df.sort_values('date').reset_index(drop=True)
print("Already sorted?", df['date'].is_monotonic_increasing)
print(df['date'].head())

Already sorted? True
0   1994-03-11
1   1994-03-11
2   1994-03-11
3   1994-03-11
4   1994-03-11
Name: date, dtype: datetime64[us]


In [7]:
# Check winner column for non-standard values
print(df['Winner'].value_counts(dropna=False))

# Check for duplicate fights (same two fighters on same date)
dupes = df.duplicated(subset=['R_fighter', 'B_fighter', 'date'], keep=False)
print(f"\nDuplicate fights: {dupes.sum()}")

Winner
Red     3979
Blue    1923
Draw     110
Name: count, dtype: int64

Duplicate fights: 2


In [8]:
# Inspect the duplicates
print(df[dupes][['R_fighter', 'B_fighter', 'date', 'Winner']])

# Drop duplicates, keep first occurrence
df = df.drop_duplicates(subset=['R_fighter', 'B_fighter', 'date'], keep='first').reset_index(drop=True)
print(f"Shape after dropping duplicates: {df.shape}")

            R_fighter        B_fighter       date Winner
131  Kazushi Sakuraba  Marcus Silveira 1997-12-21    Red
133  Kazushi Sakuraba  Marcus Silveira 1997-12-21   Draw
Shape after dropping duplicates: (6011, 144)


## Cleaning Decisions

**Duplicates (2 rows removed):** Kazushi Sakuraba vs Marcus Silveira (1997-12-21) appears twice with conflicting outcomes (Red win and Draw). This reflects a real scoring ambiguity in early PRIDE events. Kept first occurrence. Final shape: 6,011 × 144.

**Draws (110 rows):** Retained in dataset. Will be excluded at modelling stage when constructing binary win/loss labels.

**Date sorting:** Converted `date` to datetime and sorted ascending. Confirmed monotonic increasing. This is the temporal spine for all downstream feature engineering.

In [9]:
# Full null summary
null_summary = df.isnull().sum()
null_summary = null_summary[null_summary > 0].sort_values(ascending=False)
print(null_summary)
print(f"\nTotal rows: {len(df)}")

B_avg_KD                 1427
B_avg_SIG_STR_pct        1427
B_avg_opp_KD             1427
B_avg_opp_SIG_STR_pct    1427
B_avg_TD_pct             1427
                         ... 
R_Stance                   29
B_Height_cms               10
B_Weight_lbs                8
R_Height_cms                4
R_Weight_lbs                2
Length: 109, dtype: int64

Total rows: 6011


In [10]:
print(list(df.columns))

['R_fighter', 'B_fighter', 'Referee', 'date', 'location', 'Winner', 'title_bout', 'weight_class', 'B_avg_KD', 'B_avg_opp_KD', 'B_avg_SIG_STR_pct', 'B_avg_opp_SIG_STR_pct', 'B_avg_TD_pct', 'B_avg_opp_TD_pct', 'B_avg_SUB_ATT', 'B_avg_opp_SUB_ATT', 'B_avg_REV', 'B_avg_opp_REV', 'B_avg_SIG_STR_att', 'B_avg_SIG_STR_landed', 'B_avg_opp_SIG_STR_att', 'B_avg_opp_SIG_STR_landed', 'B_avg_TOTAL_STR_att', 'B_avg_TOTAL_STR_landed', 'B_avg_opp_TOTAL_STR_att', 'B_avg_opp_TOTAL_STR_landed', 'B_avg_TD_att', 'B_avg_TD_landed', 'B_avg_opp_TD_att', 'B_avg_opp_TD_landed', 'B_avg_HEAD_att', 'B_avg_HEAD_landed', 'B_avg_opp_HEAD_att', 'B_avg_opp_HEAD_landed', 'B_avg_BODY_att', 'B_avg_BODY_landed', 'B_avg_opp_BODY_att', 'B_avg_opp_BODY_landed', 'B_avg_LEG_att', 'B_avg_LEG_landed', 'B_avg_opp_LEG_att', 'B_avg_opp_LEG_landed', 'B_avg_DISTANCE_att', 'B_avg_DISTANCE_landed', 'B_avg_opp_DISTANCE_att', 'B_avg_opp_DISTANCE_landed', 'B_avg_CLINCH_att', 'B_avg_CLINCH_landed', 'B_avg_opp_CLINCH_att', 'B_avg_opp_CLINCH_l

In [11]:
# Look at a fighter with many fights to see if avg stats change fight to fight
sample = df[df['R_fighter'] == 'Jon Jones'][['date', 'R_avg_KD', 'R_avg_SIG_STR_pct', 'R_wins']].head(10)
print(sample)

           date  R_avg_KD  R_avg_SIG_STR_pct  R_wins
942  2008-08-09       NaN                NaN       0
1040 2009-01-31  0.000000           0.410000       1
1140 2009-07-11  0.500000           0.565000       2
1381 2010-08-01  0.062500           0.550625       4
1520 2011-02-05  0.031250           0.740313       5
1697 2011-09-24  0.507812           0.632578       7
1776 2011-12-10  0.253906           0.576289       8
1884 2012-04-21  0.626953           0.513145       9
2029 2012-09-22  0.313477           0.501572      10
2255 2013-04-27  0.656738           0.545786      11


In [12]:
print(list(df.columns))

['R_fighter', 'B_fighter', 'Referee', 'date', 'location', 'Winner', 'title_bout', 'weight_class', 'B_avg_KD', 'B_avg_opp_KD', 'B_avg_SIG_STR_pct', 'B_avg_opp_SIG_STR_pct', 'B_avg_TD_pct', 'B_avg_opp_TD_pct', 'B_avg_SUB_ATT', 'B_avg_opp_SUB_ATT', 'B_avg_REV', 'B_avg_opp_REV', 'B_avg_SIG_STR_att', 'B_avg_SIG_STR_landed', 'B_avg_opp_SIG_STR_att', 'B_avg_opp_SIG_STR_landed', 'B_avg_TOTAL_STR_att', 'B_avg_TOTAL_STR_landed', 'B_avg_opp_TOTAL_STR_att', 'B_avg_opp_TOTAL_STR_landed', 'B_avg_TD_att', 'B_avg_TD_landed', 'B_avg_opp_TD_att', 'B_avg_opp_TD_landed', 'B_avg_HEAD_att', 'B_avg_HEAD_landed', 'B_avg_opp_HEAD_att', 'B_avg_opp_HEAD_landed', 'B_avg_BODY_att', 'B_avg_BODY_landed', 'B_avg_opp_BODY_att', 'B_avg_opp_BODY_landed', 'B_avg_LEG_att', 'B_avg_LEG_landed', 'B_avg_opp_LEG_att', 'B_avg_opp_LEG_landed', 'B_avg_DISTANCE_att', 'B_avg_DISTANCE_landed', 'B_avg_opp_DISTANCE_att', 'B_avg_opp_DISTANCE_landed', 'B_avg_CLINCH_att', 'B_avg_CLINCH_landed', 'B_avg_opp_CLINCH_att', 'B_avg_opp_CLINCH_l

In [13]:
# Drop non-feature columns
cols_to_drop = ['Referee', 'location']
df_clean = df.drop(columns=cols_to_drop)

# Fill first-fight nulls in avg columns with 0
avg_cols = [c for c in df_clean.columns if 'avg' in c]
df_clean[avg_cols] = df_clean[avg_cols].fillna(0)

print(df_clean.shape)
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

(6011, 142)
B_total_time_fought(seconds)    1427
B_Stance                          66
B_Height_cms                      10
B_Reach_cms                      890
B_Weight_lbs                       8
R_total_time_fought(seconds)     712
R_Stance                          29
R_Height_cms                       4
R_Reach_cms                      405
R_Weight_lbs                       2
B_age                            171
R_age                             63
dtype: int64


In [14]:
# Fill total_time_fought nulls with 0
time_cols = [c for c in df_clean.columns if 'total_time_fought' in c]
df_clean[time_cols] = df_clean[time_cols].fillna(0)

# Impute reach/height/weight by weight class median
for col in ['R_Reach_cms', 'B_Reach_cms', 'R_Height_cms', 'B_Height_cms', 'R_Weight_lbs', 'B_Weight_lbs']:
    df_clean[col] = df_clean.groupby('weight_class')[col].transform(lambda x: x.fillna(x.median()))

# Impute age with overall median
df_clean['R_age'] = df_clean['R_age'].fillna(df_clean['R_age'].median())
df_clean['B_age'] = df_clean['B_age'].fillna(df_clean['B_age'].median())

# Impute stance with most common
df_clean['R_Stance'] = df_clean['R_Stance'].fillna('Orthodox')
df_clean['B_Stance'] = df_clean['B_Stance'].fillna('Orthodox')

# Verify
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

B_Reach_cms    86
dtype: int64


In [15]:
df_clean['B_Reach_cms'] = df_clean['B_Reach_cms'].fillna(df_clean['B_Reach_cms'].median())

# Final check
remaining = df_clean.isnull().sum()[df_clean.isnull().sum() > 0]
print(remaining if len(remaining) > 0 else "No nulls remaining")

No nulls remaining


## Imputation Decisions

**Rolling avg columns (1,427 nulls):** Filled with 0. These are fighters appearing for the first time with no prior UFC history. Zero is the correct value — they have no recorded knockdowns, strikes, etc.

**total_time_fought (1,427 / 712 nulls):** Filled with 0. Same logic — no prior fights means no time fought.

**Reach/Height/Weight nulls:** Imputed with median by weight class. Fighters in the same weight class have similar physical profiles, making this a more accurate proxy than the overall median.

**B_Reach_cms (86 remaining nulls):** Weight class median was unavailable for these fighters (sparse weight classes with no other reach data). Fell back to overall median. Affects <1.5% of rows — acceptable given no better reference exists. Dropping these rows would be worse.

**Age nulls (R: 63, B: 171):** Imputed with overall median. Age doesn't vary strongly enough by weight class to justify grouped imputation.

**Stance nulls (R: 29, B: 66):** Filled with 'Orthodox' — the most common stance by a significant margin in MMA.

In [16]:
# Save clean dataset
df_clean.to_csv('../data/processed/fights_clean.csv', index=False)
print(f"Saved: {df_clean.shape}")


Saved: (6011, 142)


In [17]:
print("=== CLEAN DATASET SUMMARY ===")
print(f"Shape: {df_clean.shape}")
print(f"Date range: {df_clean['date'].min()} to {df_clean['date'].max()}")
print(f"Unique fighters: {pd.concat([df_clean['R_fighter'], df_clean['B_fighter']]).nunique()}")
print(f"Weight classes: {df_clean['weight_class'].nunique()}")
print(f"\nWinner distribution:")
print(df_clean['Winner'].value_counts())
print(f"\nNulls remaining: {df_clean.isnull().sum().sum()}")

=== CLEAN DATASET SUMMARY ===
Shape: (6011, 142)
Date range: 1994-03-11 00:00:00 to 2021-03-20 00:00:00
Unique fighters: 2139
Weight classes: 14

Winner distribution:
Winner
Red     3979
Blue    1923
Draw     109
Name: count, dtype: int64

Nulls remaining: 0


## Summary

Clean dataset ready for feature engineering.

- 6,011 fights, 142 columns, 2,139 unique fighters, 14 weight classes
- Date range: 1994-03-11 to 2021-03-20
- All nulls resolved
- Sorted chronologically — temporal spine confirmed
- Draws retained (109 rows), will be excluded at modelling stage

**Class imbalance note:** Red wins 3,979 (66%) vs Blue wins 1,923 (32%). Red corner is assigned to the higher-ranked fighter by UFC convention, so this imbalance reflects real signal, not a data error. A naive "always pick Red" baseline achieves 66% accuracy — this is the floor the model must beat. Accuracy is therefore a useless metric for this problem. Primary metrics will be AUC and log-loss.

**Next step:** notebooks/02_feature_engineering.ipynb — rebuild rolling features from scratch with guaranteed no lookahead, add ELO ratings.